In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

# We set-up a simple time-independent two-well system

We start with the usual equation of the resonant flux-biased ATS Hamiltonian in the RWA
\begin{equation}
    \begin{cases}
    \dot a = -i\beta_a'a^*b\\
    \dot b = -i\beta_b'a^2/2-i\varepsilon_d'/2-\kappa_b'b
    \end{cases}
\end{equation}
Then, we perform adiabatic elimination $\dot b = 0$ and we get $$\dot a = -i\beta_a'a^*(i\beta_b'a^2/2\kappa_b'-i\varepsilon_d'/2\kappa_b')$$

We introduce the quadratures $x=(a+a^*)/\sqrt 2$ and $p=i(a^*-a)/\sqrt 2$ 
\begin{equation}
    \begin{cases}
    \dot x = - \aleph (x^2+p^2)x - \beth x\\
    \dot p = - \aleph (x^2+p^2)p + \beth p
    \end{cases},
\end{equation}
with $\aleph = \beta_a'\beta_b'/4\kappa_b'$ and $\beth = \beta_a'\varepsilon_d'/2\kappa_b'$.

Then we can define a potential $U(x,p)$ so that $\dot x = \partial_xU$ and $\dot p = \partial_pU$ :$$U(x,p) = \aleph(x^2+p^2)^2/4+\beth(x^2-p^2)/2$$



In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi*1.5

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.2
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
sqrt(2*ε_d_p/β_b_p)

In [ ]:
function equation!(du,u,params,t)
    ℵ,ℶ = params

    x = u[1]
    p = u[2]
        
    du[1] = -ℵ*(x^2+p^2)*x-ℶ*x
    du[2] = -ℵ*(x^2+p^2)*p+ℶ*p
    return nothing
end

In [ ]:
t_min = 0.0
t_max = 2/κ_b_p 
tspan = (t_min, t_max)
x_0, p_0 = 0.0, -0.001
u_0 = [x_0, p_0]
prob = ODEProblem(equation!, u_0, tspan, (ℵ,ℶ))
sol = solve(prob, Tsit5())
t = sol.t
x = [u[1] for u in sol.u]
p = [u[2] for u in sol.u]
p_stable=p;

In [ ]:
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U,
    colorbar_title="\n Meta-potential",
    xlabel=L"$x$",
    ylabel=L"$p$",
)
scatter!(x,p, label="Trajectory")

Since the system is only 1D ($x=0$ on the minimizing path and on the dissipative path), we will focus only on $p$. Therfore the equation of motion reads
$$\dot p = -\aleph p^3+\beth p$$

# Writing the Hamiltonian and initializing the GMAM

We now add noise to the system
\begin{equation}
    \dot p = - \aleph p^3 + \beth p + \sqrt \epsilon\eta(t).
\end{equation}

We can write the associated action: $$S=\int dt [P_p\dot p - H(p,P_p)],$$
with the Hamiltonian $$H(p,P_p) = [ - \aleph p^2+\beth]pP_p + \epsilon P_p^2$$

We parametrize time to solve a geometric problem $$\min_{\lambda\geq0}\min_p\max_P\int_0^1[p'P-\lambda^{-1}H(p,P)]ds,$$
and we add a constraint on the parametrization of $p$ : $|dp/ds|=|p'|=cst$

Inputs: $M = 100 \in \mathbb N$, two stable fixed points $\phi_a = (-\sqrt{2\varepsilon_d'/\beta_b'})$ and $\phi_b = (0)$ of the noiseless dynamics where $H(\phi_{a,b},0)=\nabla_{\bm \theta}H(\phi_{a,b},0)=0$; a path $\{\hat\phi_i^0\}_{i\in I}$ with $\hat\phi_0^0 =\phi_a$ and $\hat\phi_M^0 =\phi_b$, such that $|\hat\phi_{i+1}^0-\hat\phi_{i}^0|$ is constant in $i$; the functions $f(\mathbf u, \mathbf v)= \alpha \nabla_{\bm \phi}H - \nabla_{\bm \theta}H$, $g(\mathbf u, \mathbf v)=\alpha \nabla_{\bm \phi}H + \nabla_{\bm \theta}H$ and $\lambda(\mathbf u, \mathbf v) = \biggl[|\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle|+\sqrt{\max(0,\langle \nabla_{\bm \theta}H, \partial_s \hat\phi \rangle^2-4H|\partial_s\hat\phi|^2)}\biggr]/(2|\partial_s\hat\phi|^2)$; $\Delta\tau = 0.1 > 0, \alpha = 1 >0$

We have $\nabla_{\bm\phi}H = \begin{pmatrix}- 3\aleph p^2P_p +\beth P_p \end{pmatrix}$ and $\nabla_{\bm\theta}H = \begin{pmatrix} -\aleph p^3 +\beth p + 2 \epsilon P_p\end{pmatrix}$

# Let's perform the gradient descent ascent

We got that for $n\geq0$, we solve for each $i=M,M-1,\dots,1,0$ for the value of $u_i^{n+1}$ through $$u_M^{n+1} = -v_M^n+2\phi_b,$$ $$ u_i^{n+1} = \biggl[\lambda_i(u^n,v^n)u_{i+1}^{n+1}/\Delta s + f(u_i^n,v_i^n) + u_i^n/\Delta \tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^n,v^n)/\Delta s\biggr] , \qquad i=M-1,\dots,1,0$$

For the value of $v_i^{n+1}$ through $$v_0^{n+1} = -u_0^{n+1}+2\phi_a,$$ $$ v_{i}^{n+1} = \biggl[\lambda_i(u^{n+1},v^n)v_{i-1}^{n+1}/\Delta s + g(u_i^{n+1},v_i^n)+v_i^n/\Delta\tau\biggr]/\biggl[1/\Delta\tau + \lambda_i(u^{n+1},v^n)/\Delta s\biggr] , \qquad i=1,\dots,M-1,M$$

In [ ]:
M = 200                    # start smaller, then increase
ϵ = 1e-3
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))

α = 0.01
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init = θ_best#zeros(M+1)# θ_best#

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / (M)
dt = 0.1
Δτ  = 0.05*dt
N_iter = 1e4

for n in 1:N_iter
    # ---------- pass 1: build λ(u^n,v^n), update u ----------
    p  = (u .+ v) ./ 2
    Pp = (u .- v) ./ (2α)

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    fvals = @. α*gradϕ - gradθ

    lambda = zeros(Float64, M+1)

    # boundaries usually fixed / not used
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot = (p[i+1] - p[i]) / Δs

        normXdotSquare = xdot^2
        dH_dtheta = gradθ[i]
        scalarProduct = xdot * dH_dtheta
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end
    u_new = similar(u)
    u_new[end] = -v[end] + 2 * ϕ_b
    u_new[M] = (u[M] / Δτ + (lambda[M] / Δs) * u_new[M+1] + fvals[M]) / (1 / Δτ + lambda[M] / Δs)

    for i in M-1:-1:1
        u_new[i] = (u[i] / Δτ + (lambda[i] / (2 * Δs)) * (-u_new[i+2] + 4 * u_new[i+1]) + fvals[i]) /
                (1 / Δτ + 3 * lambda[i] / (2 * Δs))
    end
    p = (u_new .+ v) ./ 2
    Pp = (u_new .- v) ./ (2α)
    #p[1] = ϕ_a
    #p[end] = ϕ_b
    #Pp[1] = 0
    #Pp[end] = 0

    # s = zeros(Float64, M+1)

    # for i in 2:M+1
    #     s[i] = s[i-1] + abs(p[i] - p[i-1])
    # end

    # stringLength = s[end]

    # if stringLength > 1e-14
    #     for i in 2:M+1
    #         s[i] /= stringLength
    #     end
    # end
    # normalizedString = collect(range(0.0, 1.0, length=M+1))

    # pBis = copy(p)
    # PpBis = copy(Pp)

    # for i in 2:M
    #     lastPoint = 2   # Julia indexing version of C's lastPoint = 1

    #     for k in lastPoint:M+1
    #         if s[k-1] <= normalizedString[i] < s[k]
    #             q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

    #             pBis[i]   = (1 - q) * p[k-1]   + q * p[k]
    #             PpBis[i] = (1 - q) * Pp[k-1] + q * Pp[k]
    #             break
    #         end
    #         lastPoint = k
    #     end
    # end
    # for i in 2:M
	#     p[i] = pBis[i]
    #     Pp[i] = PpBis[i]
	# end
    u_new = @. p + α*Pp
    v = @. p - α*Pp

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    gvals = @. α*gradϕ + gradθ

    lambda = zeros(Float64, M+1)

    # boundaries usually fixed / not used
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot = (p[i+1] - p[i]) / Δs

        normXdotSquare = xdot^2
        dH_dtheta = gradθ[i]
        scalarProduct = xdot * dH_dtheta
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end

    v_new = similar(v)

    v_new[1] = -u_new[1] + 2 * ϕ_a
    v_new[2] = (v[2] / Δτ + (lambda[2] / Δs) * v_new[1] + gvals[2]) / (1 / Δτ + lambda[2] / Δs)

    @inbounds for i in 3:M+1
        λi = lambda[i]
        v_new[i] = (v[i] / Δτ + (λi / (2 * Δs)) * (4 * v_new[i-1] - v_new[i-2]) + gvals[i]) /
                   (1 / Δτ + 3 * λi / (2 * Δs))
    end

    p = (u_new .+ v_new) ./ 2
    Pp = (u_new .- v_new) ./ (2α)
    #p[1] = ϕ_a
    #p[end] = ϕ_b
    #Pp[1] = 0
    #Pp[end] = 0


    # s = zeros(Float64, M+1)

    # for i in 2:M+1
    #     s[i] = s[i-1] + abs(p[i] - p[i-1])
    # end

    # stringLength = s[end]

    # if stringLength > 1e-14
    #     for i in 2:M+1
    #         s[i] /= stringLength
    #     end
    # end
    # normalizedString = collect(range(0.0, 1.0, length=M+1))

    # pBis = copy(p)
    # PpBis = copy(Pp)

    # for i in 2:M
    #     lastPoint = 2   # Julia indexing version of C's lastPoint = 1

    #     for k in lastPoint:M+1
    #         if s[k-1] <= normalizedString[i] < s[k]
    #             q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

    #             pBis[i]   = (1 - q) * p[k-1]   + q * p[k]
    #             PpBis[i] = (1 - q) * Pp[k-1] + q * Pp[k]
    #             break
    #         end
    #         lastPoint = k
    #     end
    # end
    # for i in 2:M
	#     p[i] = pBis[i]
    #     Pp[i] = PpBis[i]
	# end
    u_new = @. p + α*Pp
    v_new = @. p - α*Pp
    u = u_new
    v = v_new
end


p = (u .+ v) ./ 2
P = (u .- v) ./ (2α)

phi_all = -.25:0.001:.01
theta_all = -.1:0.1:7
function H(p,P_p,params)
    ℵ, ℶ = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, (ℵ, ℶ)) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val, colorbar_title="\n Hamiltonian", title=L"\aleph/\epsilon = "*"$(round(ℵ/ϵ))")
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
scatter!(P,p, label="Solution of the GDA", ms=1)
#xlims!(minimum(theta_all), 3)
#ylims!(minimum(phi_all), -0.2)
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/switching_rate/issue_renormalization_toy_model_20_03_2026.pdf")

In [ ]:
M = 200                    # start smaller, then increase
ϵ = 1e-3
ϕ_a = -sqrt(2*ε_d_p/β_b_p)
ϕ_b = 0
ϕ_grid = collect(range(ϕ_a, ϕ_b, length=M+1))

α = 0.01
θ_best = @. (ℵ*ϕ_grid^3 - ℶ*ϕ_grid) / ϵ   # = -b(ϕ)/ϵ
θ_init =zeros(M+1)# θ_best#

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / (M)
dt = 0.1
Δτ  = 0.05*dt
N_iter = 1e3

for n in 1:N_iter
    # ---------- pass 1: build λ(u^n,v^n), update u ----------
    p  = (u .+ v) ./ 2
    Pp = (u .- v) ./ (2α)

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    fvals = @. α*gradϕ - gradθ

    lambda = zeros(Float64, M+1)

    # boundaries usually fixed / not used
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot = (p[i+1] - p[i]) / Δs

        normXdotSquare = xdot^2
        dH_dtheta = gradθ[i]
        scalarProduct = xdot * dH_dtheta
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end
    u_new = similar(u)
    u_new[end] = ϕ_b
    u_new[M] = (u[M] / Δτ + (lambda[M] / Δs) * u_new[M+1] + fvals[M]) / (1 / Δτ + lambda[M] / Δs)

    for i in M-1:-1:1
        u_new[i] = (u[i] / Δτ + (lambda[i] / (2 * Δs)) * (-u_new[i+2] + 4 * u_new[i+1]) + fvals[i]) /
                (1 / Δτ + 3 * lambda[i] / (2 * Δs))
    end
    p = (u_new .+ v) ./ 2
    Pp = (u_new .- v) ./ (2α)
    p[1] = ϕ_a
    p[end] = ϕ_b
    Pp[1] = 0
    Pp[end] = 0

    s = zeros(Float64, M+1)

    for i in 2:M+1
        s[i] = s[i-1] + abs(p[i] - p[i-1])
    end

    stringLength = s[end]

    if stringLength > 1e-14
        for i in 2:M+1
            s[i] /= stringLength
        end
    end
    normalizedString = collect(range(0.0, 1.0, length=M+1))

    pBis = copy(p)
    PpBis = copy(Pp)

    for i in 2:M
        lastPoint = 2   # Julia indexing version of C's lastPoint = 1

        for k in lastPoint:M+1
            if s[k-1] <= normalizedString[i] < s[k]
                q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

                pBis[i]   = (1 - q) * p[k-1]   + q * p[k]
                PpBis[i] = (1 - q) * Pp[k-1] + q * Pp[k]
                break
            end
            lastPoint = k
        end
    end
    for i in 2:M
	    p[i] = pBis[i]
        Pp[i] = PpBis[i]
	end
    u_new = @. p + α*Pp
    v = @. p - α*Pp

    gradθ = @. (-ℵ*p^3 + ℶ*p) + 2*ϵ*Pp
    gradϕ = @. (-3*ℵ*p^2 + ℶ) * Pp
    Hvals = @. ((-ℵ*p^3 + ℶ*p) * Pp) + ϵ*Pp^2
    gvals = @. α*gradϕ + gradθ

    lambda = zeros(Float64, M+1)

    # boundaries usually fixed / not used
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot = (p[i+1] - p[i]) / Δs

        normXdotSquare = xdot^2
        dH_dtheta = gradθ[i]
        scalarProduct = xdot * dH_dtheta
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end

    v_new = similar(v)

    v_new[1] = ϕ_a
    v_new[2] = (v[2] / Δτ + (lambda[2] / Δs) * v_new[1] + gvals[2]) / (1 / Δτ + lambda[2] / Δs)

    @inbounds for i in 3:M+1
        λi = lambda[i]
        v_new[i] = (v[i] / Δτ + (λi / (2 * Δs)) * (4 * v_new[i-1] - v_new[i-2]) + gvals[i]) /
                   (1 / Δτ + 3 * λi / (2 * Δs))
    end

    p = (u_new .+ v_new) ./ 2
    Pp = (u_new .- v_new) ./ (2α)
    p[1] = ϕ_a
    p[end] = ϕ_b
    Pp[1] = 0
    Pp[end] = 0


    s = zeros(Float64, M+1)

    for i in 2:M+1
        s[i] = s[i-1] + abs(p[i] - p[i-1])
    end

    stringLength = s[end]

    if stringLength > 1e-14
        for i in 2:M+1
            s[i] /= stringLength
        end
    end
    normalizedString = collect(range(0.0, 1.0, length=M+1))

    pBis = copy(p)
    PpBis = copy(Pp)

    for i in 2:M
        lastPoint = 2   # Julia indexing version of C's lastPoint = 1

        for k in lastPoint:M+1
            if s[k-1] <= normalizedString[i] < s[k]
                q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

                pBis[i]   = (1 - q) * p[k-1]   + q * p[k]
                PpBis[i] = (1 - q) * Pp[k-1] + q * Pp[k]
                break
            end
            lastPoint = k
        end
    end
    for i in 2:M
	    p[i] = pBis[i]
        Pp[i] = PpBis[i]
	end
    u_new = @. p + α*Pp
    v_new = @. p - α*Pp
    u = u_new
    v = v_new
end


p = (u .+ v) ./ 2
P = (u .- v) ./ (2α)

phi_all = -.25:0.001:.01
theta_all = -.1:0.1:7
function H(p,P_p,params)
    ℵ, ℶ = params
    return P_p*(-ℵ*(p^2)*p + ℶ*p) +
           ϵ*P_p^2
end
H_val = [H(p, P_p, (ℵ, ℶ)) for p in phi_all, P_p in theta_all]


heatmap(theta_all,phi_all, H_val, colorbar_title="\n Hamiltonian", title=L"\aleph/\epsilon = "*"$(round(ℵ/ϵ))")
xlabel!(L"$P_p$")
ylabel!(L"$p$")
contour!(theta_all, phi_all, H_val; levels=[0.0], c=:white, lw=2)
plot!(zeros(length(p_stable)), p_stable, c="red", ls=:dash, label="Noise-free dissipation")
plot!(θ_best, ϕ_grid, label="Optimal path", ls=:dash)
scatter!(P,p, label="Solution of the GDA", ms=1)
#xlims!(minimum(theta_all), 3)
#ylims!(minimum(phi_all), -0.2)
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/switching_rate/issue_renormalization_toy_model_with_renormalization_20_03_2026.pdf")

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi /2

#Environment
κ_b=5/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.1
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
M = 400
ϵ = 3e-3

ϕ_a = [-sqrt(2 * ε_d_p / β_b_p), 0.0]
ϕ_b = [0.0, -sqrt(0.5) * ε_d_p / κ_b_p]

params = (ϵ, β_a_p, β_b_p, ε_d_p, κ_b_p)

# initial string in phi-space
ϕ_grid = zeros(M + 1, 2)
for i in 1:M+1
    t = (i - 1) / M
    ϕ_grid[i, 1] = (1 - t) * ϕ_a[1] + t * ϕ_b[1]
    ϕ_grid[i, 2] = (1 - t) * ϕ_a[2] + t * ϕ_b[2]
end

α = 0.01

# initial theta = 0
θ_init = zeros(M + 1, 2)

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / M
dt = 0.1
Δτ = 0.5 * dt
N_iter = 5e3

for n in 1:N_iter

    # =========================================================
    # PASS 1: build lambda(u^n,v^n), update u from right boundary
    # =========================================================
    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2α)

    gradθ = zeros(M + 1, 2)
    gradϕ = zeros(M + 1, 2)
    Hvals = zeros(M + 1)

    for i in 1:M+1
        p_a = ϕ[i, 1]
        p_b = ϕ[i, 2]
        P_p_a = θ[i, 1]
        P_p_b = θ[i, 2]

        gradϕ[i, 1] = -β_a_p * (p_b * P_p_a) / sqrt(2) + 2 * β_b_p * (p_a * P_p_b) / sqrt(2)
        gradϕ[i, 2] = -β_a_p * (p_a * P_p_a) / sqrt(2) - 2 * κ_b_p * P_p_b

        gradθ[i, 1] = -β_a_p * (p_a * p_b) / sqrt(2)
        gradθ[i, 2] = 2 * β_b_p * p_a^2 / (2 * sqrt(2)) - 2 * ε_d_p / sqrt(2) - 2 * κ_b_p * p_b + 2 * ϵ * P_p_b

        Hvals[i] = (-β_a_p * (p_a * p_b) / sqrt(2)) * P_p_a +
                   2 * (-β_b_p * (-p_a^2) / (2 * sqrt(2)) - ε_d_p / sqrt(2) - κ_b_p * p_b) * P_p_b +
                   ϵ * (P_p_b^2)
    end

    fvals = α .* gradϕ .- gradθ

    lambda = zeros(M + 1)
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot1 = (ϕ[i+1, 1] - ϕ[i, 1]) / Δs
        xdot2 = (ϕ[i+1, 2] - ϕ[i, 2]) / Δs

        normXdotSquare = xdot1^2 + xdot2^2
        scalarProduct = xdot1 * gradθ[i, 1] + xdot2 * gradθ[i, 2]
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end

    u_new = similar(u)

    # right boundary condition
    u_new[end, :] .= -v[end, :] .+ 2 .* ϕ_b

    # first-order point next to boundary
    u_new[M, :] .= (u[M, :] ./ Δτ .+ (lambda[M] / Δs) .* u_new[M+1, :] .+ fvals[M, :]) ./
                   (1 / Δτ + lambda[M] / Δs)

    # second-order interior sweep
    for i in M-1:-1:1
        u_new[i, :] .= (u[i, :] ./ Δτ .+
                        (lambda[i] / (2 * Δs)) .* (-u_new[i+2, :] .+ 4 .* u_new[i+1, :]) .+
                        fvals[i, :]) ./
                       (1 / Δτ + 3 * lambda[i] / (2 * Δs))
    end

    # reconstruct phi,theta after U update
    ϕ = (u_new .+ v) ./ 2
    θ = (u_new .- v) ./ (2α)

    ϕ[1, :] .= ϕ_a
    ϕ[end, :] .= ϕ_b
    θ[1, :] .= 0.0
    θ[end, :] .= 0.0

    # =========================================================
    # REPARAMETRIZATION after U step
    # =========================================================
    s = zeros(M + 1)
    for i in 2:M+1
        d1 = ϕ[i, 1] - ϕ[i-1, 1]
        d2 = ϕ[i, 2] - ϕ[i-1, 2]
        s[i] = s[i-1] + sqrt(d1^2 + d2^2)
    end

    stringLength = s[end]
    if stringLength > 1e-14
        for i in 2:M+1
            s[i] /= stringLength
        end
    end

    normalizedString = collect(range(0.0, 1.0, length=M+1))

    ϕBis = copy(ϕ)
    θBis = copy(θ)

    for i in 2:M
        lastPoint = 2
        for k in lastPoint:M+1
            if s[k-1] <= normalizedString[i] < s[k]
                q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

                ϕBis[i, :] .= (1 - q) .* ϕ[k-1, :] .+ q .* ϕ[k, :]
                θBis[i, :] .= (1 - q) .* θ[k-1, :] .+ q .* θ[k, :]
                break
            end
            lastPoint = k
        end
    end

    for i in 2:M
        ϕ[i, :] .= ϕBis[i, :]
        θ[i, :] .= θBis[i, :]
    end

    u_new = ϕ .+ α .* θ
    v = ϕ .- α .* θ

    # =========================================================
    # PASS 2: build lambda(u^{n+1},v^n), update v from left boundary
    # =========================================================
    ϕ = (u_new .+ v) ./ 2
    θ = (u_new .- v) ./ (2α)

    gradθ .= 0.0
    gradϕ .= 0.0
    Hvals .= 0.0

    for i in 1:M+1
        p_a = ϕ[i, 1]
        p_b = ϕ[i, 2]
        P_p_a = θ[i, 1]
        P_p_b = θ[i, 2]

        gradϕ[i, 1] = -β_a_p * (p_b * P_p_a) / sqrt(2) + 2 * β_b_p * (p_a * P_p_b) / sqrt(2)
        gradϕ[i, 2] = -β_a_p * (p_a * P_p_a) / sqrt(2) - 2 * κ_b_p * P_p_b

        gradθ[i, 1] = -β_a_p * (p_a * p_b) / sqrt(2)
        gradθ[i, 2] = 2 * β_b_p * p_a^2 / (2 * sqrt(2)) - 2 * ε_d_p / sqrt(2) - 2 * κ_b_p * p_b + 2 * ϵ * P_p_b

        Hvals[i] = (-β_a_p * (p_a * p_b) / sqrt(2)) * P_p_a +
                   2 * (-β_b_p * (-p_a^2) / (2 * sqrt(2)) - ε_d_p / sqrt(2) - κ_b_p * p_b) * P_p_b +
                   ϵ * (P_p_b^2)
    end

    gvals = α .* gradϕ .+ gradθ

    lambda .= 0.0
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot1 = (ϕ[i+1, 1] - ϕ[i, 1]) / Δs
        xdot2 = (ϕ[i+1, 2] - ϕ[i, 2]) / Δs

        normXdotSquare = xdot1^2 + xdot2^2
        scalarProduct = xdot1 * gradθ[i, 1] + xdot2 * gradθ[i, 2]
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end

    v_new = similar(v)

    # left boundary condition
    v_new[1, :] .= -u_new[1, :] .+ 2 .* ϕ_a

    # first-order point next to boundary
    v_new[2, :] .= (v[2, :] ./ Δτ .+ (lambda[2] / Δs) .* v_new[1, :] .+ gvals[2, :]) ./
                   (1 / Δτ + lambda[2] / Δs)

    # second-order interior sweep
    for i in 3:M+1
        λi = lambda[i]
        v_new[i, :] .= (v[i, :] ./ Δτ .+
                        (λi / (2 * Δs)) .* (4 .* v_new[i-1, :] .- v_new[i-2, :]) .+
                        gvals[i, :]) ./
                       (1 / Δτ + 3 * λi / (2 * Δs))
    end

    # reconstruct phi,theta after V update
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    ϕ[1, :] .= ϕ_a
    ϕ[end, :] .= ϕ_b
    θ[1, :] .= 0.0
    θ[end, :] .= 0.0

    # =========================================================
    # REPARAMETRIZATION after V step
    # =========================================================
    s .= 0.0
    for i in 2:M+1
        d1 = ϕ[i, 1] - ϕ[i-1, 1]
        d2 = ϕ[i, 2] - ϕ[i-1, 2]
        s[i] = s[i-1] + sqrt(d1^2 + d2^2)
    end

    stringLength = s[end]
    if stringLength > 1e-14
        for i in 2:M+1
            s[i] /= stringLength
        end
    end

    ϕBis .= ϕ
    θBis .= θ

    for i in 2:M
        lastPoint = 2
        for k in lastPoint:M+1
            if s[k-1] <= normalizedString[i] < s[k]
                q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

                ϕBis[i, :] .= (1 - q) .* ϕ[k-1, :] .+ q .* ϕ[k, :]
                θBis[i, :] .= (1 - q) .* θ[k-1, :] .+ q .* θ[k, :]
                break
            end
            lastPoint = k
        end
    end

    for i in 2:M
        ϕ[i, :] .= ϕBis[i, :]
        θ[i, :] .= θBis[i, :]
    end

    u = ϕ .+ α .* θ
    v = ϕ .- α .* θ
end

ϕ = (u .+ v) ./ 2
θ = (u .- v) ./ (2α)

p_a = ϕ[:, 1]
p_b = ϕ[:, 2]
P_p_a = θ[:, 1]
P_p_b = θ[:, 2]

In [ ]:
# --- subplot 1 ---
p1 = plot(P_p_a, p_a,
    xlabel=L"$P_{p_a}$",
    ylabel=L"$p_a$",
     label="GDA")

# --- subplot 2 ---
p2 = plot(P_p_b, p_b,
    xlabel=L"$P_{p_b}$",
    ylabel=L"$p_b$",
    label="GDA")

# --- combine ---
plot(p1, p2, layout=(1,2))

In [ ]:
E_J = 37.12*2pi

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.5
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
M = 500
ϵ = 1e-3

ϕ_a = [-sqrt(2 * ε_d_p / β_b_p), 0.0]
ϕ_b = [0.0, -sqrt(0.5) * ε_d_p / κ_b_p]

params = (ϵ, β_a_p, β_b_p, ε_d_p, κ_b_p)

# initial string in phi-space
ϕ_grid = zeros(M + 1, 2)
for i in 1:M+1
    t = (i - 1) / M
    ϕ_grid[i, 1] = (1 - t) * ϕ_a[1] + t * ϕ_b[1]
    ϕ_grid[i, 2] = (1 - t) * ϕ_a[2] + t * ϕ_b[2]
end

α = 0.01

# initial theta = 0
θ_init = zeros(M + 1, 2)

u = ϕ_grid .+ α .* θ_init
v = ϕ_grid .- α .* θ_init

Δs = 1 / M
dt = 0.1
Δτ = 0.05 * dt
N_iter = 1000

for n in 1:N_iter

    # =========================================================
    # PASS 1: build lambda(u^n,v^n), update u from right boundary
    # =========================================================
    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2α)

    gradθ = zeros(M + 1, 2)
    gradϕ = zeros(M + 1, 2)
    Hvals = zeros(M + 1)

    for i in 1:M+1
        p_a = ϕ[i, 1]
        p_b = ϕ[i, 2]
        P_p_a = θ[i, 1]
        P_p_b = θ[i, 2]

        gradϕ[i, 1] = -β_a_p * (p_b * P_p_a) / sqrt(2) + 2 * β_b_p * (p_a * P_p_b) / sqrt(2)
        gradϕ[i, 2] = -β_a_p * (p_a * P_p_a) / sqrt(2) - 2 * κ_b_p * P_p_b

        gradθ[i, 1] = -β_a_p * (p_a * p_b) / sqrt(2)
        gradθ[i, 2] = 2 * β_b_p * p_a^2 / (2 * sqrt(2)) - 2 * ε_d_p / sqrt(2) - 2 * κ_b_p * p_b + 2 * ϵ * P_p_b

        Hvals[i] = (-β_a_p * (p_a * p_b) / sqrt(2)) * P_p_a +
                   2 * (-β_b_p * (-p_a^2) / (2 * sqrt(2)) - ε_d_p / sqrt(2) - κ_b_p * p_b) * P_p_b +
                   ϵ * (P_p_b^2)
    end

    fvals = α .* gradϕ .- gradθ

    lambda = zeros(M + 1)
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot1 = (ϕ[i+1, 1] - ϕ[i, 1]) / Δs
        xdot2 = (ϕ[i+1, 2] - ϕ[i, 2]) / Δs

        normXdotSquare = xdot1^2 + xdot2^2
        scalarProduct = xdot1 * gradθ[i, 1] + xdot2 * gradθ[i, 2]
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end

    u_new = similar(u)

    # right boundary condition
    u_new[end, :] .= -v[end, :] .+ 2 .* ϕ_b

    # first-order point next to boundary
    u_new[M, :] .= (u[M, :] ./ Δτ .+ (lambda[M] / Δs) .* u_new[M+1, :] .+ fvals[M, :]) ./
                   (1 / Δτ + lambda[M] / Δs)

    # second-order interior sweep
    for i in M-1:-1:1
        u_new[i, :] .= (u[i, :] ./ Δτ .+
                        (lambda[i] / (2 * Δs)) .* (-u_new[i+2, :] .+ 4 .* u_new[i+1, :]) .+
                        fvals[i, :]) ./
                       (1 / Δτ + 3 * lambda[i] / (2 * Δs))
    end

    # reconstruct phi,theta after U update
    ϕ = (u_new .+ v) ./ 2
    θ = (u_new .- v) ./ (2α)

    ϕ[1, :] .= ϕ_a
    ϕ[end, :] .= ϕ_b
    θ[1, :] .= 0.0
    θ[end, :] .= 0.0

    # =========================================================
    # REPARAMETRIZATION after U step
    # =========================================================
    s = zeros(M + 1)
    for i in 2:M+1
        d1 = ϕ[i, 1] - ϕ[i-1, 1]
        d2 = ϕ[i, 2] - ϕ[i-1, 2]
        s[i] = s[i-1] + sqrt(d1^2 + d2^2)
    end

    stringLength = s[end]
    if stringLength > 1e-14
        for i in 2:M+1
            s[i] /= stringLength
        end
    end

    normalizedString = collect(range(0.0, 1.0, length=M+1))

    ϕBis = copy(ϕ)
    θBis = copy(θ)

    for i in 2:M
        lastPoint = 2
        for k in lastPoint:M+1
            if s[k-1] <= normalizedString[i] < s[k]
                q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

                ϕBis[i, :] .= (1 - q) .* ϕ[k-1, :] .+ q .* ϕ[k, :]
                θBis[i, :] .= (1 - q) .* θ[k-1, :] .+ q .* θ[k, :]
                break
            end
            lastPoint = k
        end
    end

    for i in 2:M
        ϕ[i, :] .= ϕBis[i, :]
        θ[i, :] .= θBis[i, :]
    end

    u_new = ϕ .+ α .* θ
    v = ϕ .- α .* θ

    # =========================================================
    # PASS 2: build lambda(u^{n+1},v^n), update v from left boundary
    # =========================================================
    ϕ = (u_new .+ v) ./ 2
    θ = (u_new .- v) ./ (2α)

    gradθ .= 0.0
    gradϕ .= 0.0
    Hvals .= 0.0

    for i in 1:M+1
        p_a = ϕ[i, 1]
        p_b = ϕ[i, 2]
        P_p_a = θ[i, 1]
        P_p_b = θ[i, 2]

        gradϕ[i, 1] = -β_a_p * (p_b * P_p_a) / sqrt(2) + 2 * β_b_p * (p_a * P_p_b) / sqrt(2)
        gradϕ[i, 2] = -β_a_p * (p_a * P_p_a) / sqrt(2) - 2 * κ_b_p * P_p_b

        gradθ[i, 1] = -β_a_p * (p_a * p_b) / sqrt(2)
        gradθ[i, 2] = 2 * β_b_p * p_a^2 / (2 * sqrt(2)) - 2 * ε_d_p / sqrt(2) - 2 * κ_b_p * p_b + 2 * ϵ * P_p_b

        Hvals[i] = (-β_a_p * (p_a * p_b) / sqrt(2)) * P_p_a +
                   2 * (-β_b_p * (-p_a^2) / (2 * sqrt(2)) - ε_d_p / sqrt(2) - κ_b_p * p_b) * P_p_b +
                   ϵ * (P_p_b^2)
    end

    gvals = α .* gradϕ .+ gradθ

    lambda .= 0.0
    lambda[1] = 0.0
    lambda[end] = 0.0

    for i in 2:M
        xdot1 = (ϕ[i+1, 1] - ϕ[i, 1]) / Δs
        xdot2 = (ϕ[i+1, 2] - ϕ[i, 2]) / Δs

        normXdotSquare = xdot1^2 + xdot2^2
        scalarProduct = xdot1 * gradθ[i, 1] + xdot2 * gradθ[i, 2]
        Hval = Hvals[i]
        disc = scalarProduct^2 - 4.0 * Hval * normXdotSquare

        lambda[i] = (scalarProduct + sqrt(max(0.0, disc))) /
                    (2.0 * max(normXdotSquare, 1e-14))
    end

    v_new = similar(v)

    # left boundary condition
    v_new[1, :] .= -u_new[1, :] .+ 2 .* ϕ_a

    # first-order point next to boundary
    v_new[2, :] .= (v[2, :] ./ Δτ .+ (lambda[2] / Δs) .* v_new[1, :] .+ gvals[2, :]) ./
                   (1 / Δτ + lambda[2] / Δs)

    # second-order interior sweep
    for i in 3:M+1
        λi = lambda[i]
        v_new[i, :] .= (v[i, :] ./ Δτ .+
                        (λi / (2 * Δs)) .* (4 .* v_new[i-1, :] .- v_new[i-2, :]) .+
                        gvals[i, :]) ./
                       (1 / Δτ + 3 * λi / (2 * Δs))
    end

    # reconstruct phi,theta after V update
    ϕ = (u_new .+ v_new) ./ 2
    θ = (u_new .- v_new) ./ (2α)

    ϕ[1, :] .= ϕ_a
    ϕ[end, :] .= ϕ_b
    θ[1, :] .= 0.0
    θ[end, :] .= 0.0

    # =========================================================
    # REPARAMETRIZATION after V step
    # =========================================================
    s .= 0.0
    for i in 2:M+1
        d1 = ϕ[i, 1] - ϕ[i-1, 1]
        d2 = ϕ[i, 2] - ϕ[i-1, 2]
        s[i] = s[i-1] + sqrt(d1^2 + d2^2)
    end

    stringLength = s[end]
    if stringLength > 1e-14
        for i in 2:M+1
            s[i] /= stringLength
        end
    end

    ϕBis .= ϕ
    θBis .= θ

    for i in 2:M
        lastPoint = 2
        for k in lastPoint:M+1
            if s[k-1] <= normalizedString[i] < s[k]
                q = (normalizedString[i] - s[k-1]) / (s[k] - s[k-1])

                ϕBis[i, :] .= (1 - q) .* ϕ[k-1, :] .+ q .* ϕ[k, :]
                θBis[i, :] .= (1 - q) .* θ[k-1, :] .+ q .* θ[k, :]
                break
            end
            lastPoint = k
        end
    end

    for i in 2:M
        ϕ[i, :] .= ϕBis[i, :]
        θ[i, :] .= θBis[i, :]
    end

    u = ϕ .+ α .* θ
    v = ϕ .- α .* θ
end

ϕ = (u .+ v) ./ 2
θ = (u .- v) ./ (2α)

p_a = ϕ[:, 1]
p_b = ϕ[:, 2]
P_p_a = θ[:, 1]
P_p_b = θ[:, 2]

In [ ]:
# --- subplot 1 ---
p1 = plot(P_p_a, p_a,
    xlabel=L"$P_{p_a}$",
    ylabel=L"$p_a$",
     label="GDA")

# --- subplot 2 ---
p2 = plot(P_p_b, p_b,
    xlabel=L"$P_{p_b}$",
    ylabel=L"$p_b$",
    label="GDA")

# --- combine ---
plot(p1, p2, layout=(1,2))